# CrazeAI — Google Colab training and GGUF export

This notebook checks out the `test` branch, trains CrazeAI from scratch using the repository's JAX pipeline, exports the checkpoint to Hugging Face format, converts it to GGUF, quantizes to `Q4_K_M`, enforces a strict **100,000,000-byte (100 MB)** limit, and downloads the result.

**Before running:** in Colab choose **Runtime → Change runtime type → T4 GPU** (or another GPU). Training from scratch is compute-intensive; a GPU is strongly recommended. This does not guarantee good assistant quality—the small example count is for a pipeline test.

In [ ]:
# Confirm runtime hardware before installing anything.
import os, sys, subprocess
print(sys.version)
try:
    import jax
    print('JAX:', jax.__version__)
    print('Backend:', jax.default_backend())
    print('Devices:', jax.devices())
except Exception as e:
    print('JAX import issue:', repr(e))
    raise
if jax.default_backend() != 'gpu':
    print('\nWARNING: JAX is not using a GPU. In Colab, select Runtime > Change runtime type > T4 GPU, reconnect, and rerun this cell.')

In [ ]:
# Clone the working branch and install dependencies without deliberately replacing Colab's JAX build.
%cd /content
!rm -rf CrazeAi llama.cpp
!git clone --depth 1 --branch test https://github.com/CrazeStudio/CrazeAi.git
%cd /content/CrazeAi
%pip install -q optax safetensors tokenizers datasets huggingface_hub
import jax, optax, numpy, tokenizers, datasets
print('Backend:', jax.default_backend(), '| Devices:', jax.devices())

In [ ]:
# Set a manageable first run. Increase MAX_EXAMPLES and EPOCHS only after the pipeline succeeds.
MAX_EXAMPLES = 1000
EPOCHS = 1
SEQ_LEN = 128
BATCH_SIZE = 1
CHUNK_SIZE = 1
MAX_MODEL_BYTES = 100_000_000

import os
os.environ['PYTHONUNBUFFERED'] = '1'
!mkdir -p artifacts
!python training_v2/prepare_sft.py --max-examples {MAX_EXAMPLES} --output artifacts/sft.jsonl
!test -s artifacts/sft.jsonl && wc -l artifacts/sft.jsonl

In [ ]:
# Train CrazeAI from scratch. This can take a while; keep this Colab session open.
!python training_jax/train.py --data artifacts/sft.jsonl --output artifacts/crazeai-jax --epochs {EPOCHS} --seq-len {SEQ_LEN} --batch-size {BATCH_SIZE} --max-examples {MAX_EXAMPLES} --chunk-size {CHUNK_SIZE}
!test -s artifacts/crazeai-jax/params.npz

In [ ]:
# Export and validate the Hugging Face checkpoint.
!python training_jax/export_hf.py --checkpoint artifacts/crazeai-jax --output artifacts/crazeai-135m
!test -s artifacts/crazeai-135m/model.safetensors
!test -s artifacts/crazeai-135m/config.json
!ls -lh artifacts/crazeai-135m/

In [ ]:
# Build llama.cpp converter/quantizer.
%cd /content
!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
!cmake -S llama.cpp -B llama.cpp/build -DCMAKE_BUILD_TYPE=Release -DGGML_NATIVE=OFF
!cmake --build llama.cpp/build --config Release -j2 --target llama-quantize llama-cli
%pip install -q -r /content/llama.cpp/requirements.txt

In [ ]:
# Convert to F16 GGUF, then quantize to Q4_K_M.
%cd /content/CrazeAi
!python /content/llama.cpp/convert_hf_to_gguf.py artifacts/crazeai-135m --outfile artifacts/crazeai-135m-f16.gguf --outtype f16
!test -s artifacts/crazeai-135m-f16.gguf
!/content/llama.cpp/build/bin/llama-quantize artifacts/crazeai-135m-f16.gguf artifacts/crazeai-135m-q4_k_m.gguf Q4_K_M
!test -s artifacts/crazeai-135m-q4_k_m.gguf
import os
model_path = '/content/CrazeAi/artifacts/crazeai-135m-q4_k_m.gguf'
size_bytes = os.path.getsize(model_path)
print(f'GGUF size: {size_bytes:,} bytes ({size_bytes / 1_000_000:.2f} MB)')
if size_bytes > MAX_MODEL_BYTES:
    raise RuntimeError(f'Model exceeds 100 MB: {size_bytes:,} bytes. Do not label this export as under 100 MB.')
print('PASS: final GGUF is within the strict 100 MB limit.')

In [ ]:
# Smoke test: generate a short response from the quantized model.
! /content/llama.cpp/build/bin/llama-cli -m /content/CrazeAi/artifacts/crazeai-135m-q4_k_m.gguf -p 'Who are you?' -n 48 --temp 0.2 --no-display-prompt

In [ ]:
# Download the model to your device. Keep the Colab tab open until download begins.
from google.colab import files
files.download('/content/CrazeAi/artifacts/crazeai-135m-q4_k_m.gguf')

## Notes
- The notebook trains from random initialization; it is not a pretrained 135M model. A short run mainly verifies that the pipeline works.
- The output-size cell fails if the final GGUF exceeds 100,000,000 bytes.
- Colab sessions can disconnect or run out of RAM/VRAM. If that happens, reduce `MAX_EXAMPLES` or `SEQ_LEN`, or resume in a fresh session after saving `/content/CrazeAi/artifacts` to Google Drive.
- Use only data you are permitted to train on.